# Block detector training (YOLOv8, Colab T4)

Objective 1.1: find the 3 blocks in one photo. Trained because the classical detector (`block_detect.py`)
failed on the rig mock-up photos: shadows in the gaps merged the blocks (0 of 18 photos).

**Before running:** Runtime > Change runtime type > **T4 GPU**. Then run the cells top to bottom.

**Data:** zip the Roboflow export folder (`CBP Block Detection.v3i.yolov8`, YOLOv8 format) on your PC
(right-click > Compress to ZIP) and upload it in cell 3. Export **without augmentation**: augmented copies are not new photos.

**Honest limits:** 18 photos is very few. The validation numbers will be noisy (6 photos).
The boxes are axis-aligned, so crops of tilted blocks include background.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

## 3. Upload the dataset zip

Option A (below): upload from your PC. Option B: put the zip in Google Drive and set `ZIP` to its path instead.

In [ ]:
import os, zipfile, glob
from google.colab import files

# Option B: from Drive
# from google.colab import drive; drive.mount('/content/drive')
# ZIP = '/content/drive/MyDrive/path/to/CBP Block Detection.v3i.yolov8.zip'

ZIP = list(files.upload().keys())[0]          # Option A
DATA = '/content/blocks_data'
zipfile.ZipFile(ZIP).extractall(DATA)
images = sorted(glob.glob(f'{DATA}/**/images/*.*', recursive=True))
print(len(images), 'images found')
assert images, 'no */images/* folder in the zip: is it a YOLOv8 export?'

## 4. Split train / validation

Split by **placement**: every `*_P003_*` photo (top and back view) goes to validation, so no placement is in both sets.
Change `VAL_KEY` if your file names change. Roboflow's own split is ignored.

In [ ]:
VAL_KEY = '_P003_'
train = [p for p in images if VAL_KEY not in os.path.basename(p)]
val   = [p for p in images if VAL_KEY in os.path.basename(p)]
print(len(train), 'train |', len(val), 'val')
assert train and val, 'empty split: check VAL_KEY against the file names'

open(f'{DATA}/train.txt', 'w').write('\n'.join(train))
open(f'{DATA}/val.txt', 'w').write('\n'.join(val))
open(f'{DATA}/blocks.yaml', 'w').write(
    f"train: {DATA}/train.txt\nval: {DATA}/val.txt\nnames: ['pavement_block']\n")
print(open(f'{DATA}/blocks.yaml').read())

## 5. Train

`yolov8n` = smallest and fastest (good for the mini PC). Try `yolov8s.pt` if `n` misses blocks.
`imgsz=1024` keeps block edges sharp. Early stopping (`patience`) ends it if validation stops improving.

In [ ]:
from ultralytics import YOLO

MODEL = 'yolov8n.pt'
model = YOLO(MODEL)
model.train(data=f'{DATA}/blocks.yaml', imgsz=1024, epochs=100, patience=30, batch=8,
            device=0, seed=0, project='/content/runs', name='blocks', exist_ok=True)
RUN = str(model.trainer.save_dir)   # actual output folder (ultralytics may nest it)
print('saved to', RUN)

## 6. Validation metrics

In [ ]:
best = YOLO(f'{RUN}/weights/best.pt')
m = best.val(data=f'{DATA}/blocks.yaml', imgsz=1024, device=0)
print(f'mAP50 {m.box.map50:.3f} | mAP50-95 {m.box.map:.3f} | precision {m.box.mp:.3f} | recall {m.box.mr:.3f}')

## 7. The check that matters for the rig: exactly 3 blocks per photo

mAP is not the pass rule. The pipeline needs **all 3 blocks found, no extras**, each box close to the annotation
(IoU >= 0.8, same rule as `block_detect.py --yolo_dir`).

In [ ]:
import numpy as np, cv2

def iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0])); ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = iw * ih
    return i / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - i + 1e-9)

def label_boxes(img_path, W, H):
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    out = []
    for line in open(lp).read().split('\n'):
        p = line.split()
        if len(p) == 5:
            cx, cy, w, h = map(float, p[1:])
            out.append(((cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H))
    return out

CONF, N = 0.5, 3
passed = 0
for p in val:
    r = best.predict(p, conf=CONF, imgsz=1024, verbose=False)[0]
    H, W = r.orig_shape
    det = r.boxes.xyxy.cpu().numpy().tolist()
    gt = label_boxes(p, W, H)
    worst = min((max((iou(g, d) for d in det), default=0) for g in gt), default=0)
    ok = len(det) == N and len(gt) == N and worst >= 0.8
    passed += ok
    print(f"[{'PASS' if ok else 'FAIL'}] {os.path.basename(p)[:20]}: found {len(det)}/{len(gt)}, worst IoU {worst:.2f}")
print(f'{passed} of {len(val)} validation photos: exactly {N} blocks, IoU >= 0.8')

## 8. Look at the predictions (and the crops, sorted Left / Middle / Right)

In [ ]:
import matplotlib.pyplot as plt
for p in val:
    r = best.predict(p, conf=CONF, imgsz=1024, verbose=False)[0]
    boxes = sorted(r.boxes.xyxy.cpu().numpy().astype(int).tolist(), key=lambda b: b[0])
    fig, ax = plt.subplots(1, 1 + len(boxes), figsize=(16, 3.5))
    ax = np.atleast_1d(ax)
    ax[0].imshow(r.plot()[..., ::-1]); ax[0].set_title(os.path.basename(p)[:20])
    names = ['Left', 'Middle', 'Right'] if len(boxes) == 3 else [f'Block {i+1}' for i in range(len(boxes))]
    for a, (x1, y1, x2, y2), n in zip(ax[1:], boxes, names):
        a.imshow(r.orig_img[y1:y2, x1:x2, ::-1]); a.set_title(n)
    for a in ax: a.axis('off')
    plt.show()

## 9. Save the weights

Download `best.pt` and put it in the repo's `models/` folder (weights are not committed to git).
Also keep `results.csv` and the versions printed below with the run.

In [ ]:
import shutil, sys
shutil.make_archive('/content/blocks_run', 'zip', RUN)
print(sys.version.split()[0], '| ultralytics', ultralytics.__version__, '| torch', torch.__version__,
      '| GPU', torch.cuda.get_device_name(0))
files.download(f'{RUN}/weights/best.pt')
files.download('/content/blocks_run.zip')     # curves, confusion matrix, results.csv, args.yaml